In [1]:
import os
import yaml
import re
import pandas as pd
from pathlib import Path
import openpyxl

In [7]:
# downloads = Path.home() / "Downloads"
downloads = Path.home() / "/Users/danielbhuglah/Library/CloudStorage/OneDrive-TheOpenUniversity/SXS841 OSO Observations/2026_02_24_files_consolidated"

In [3]:
def load_yaml(path):
    """Safe YAML loader."""
    try:
        with open(path, "r") as f:
            return yaml.safe_load(f)
    except:
        return {}



In [4]:
def extract_star_info_from_log(log_path, run_name):
    """Extracts target + comparison star info from a HOPS log.yaml file."""
    data = load_yaml(log_path)
    rows = []

    # -------------------------
    # Extract TARGET star
    # -------------------------
    target = {
        "run": run_name,
        "star_type": "target",
        "star_id": "target",
        "x": data.get("target_x_position"),
        "y": data.get("target_y_position"),
        "aperture": data.get("target_aperture"),
        "r_position": data.get("target_r_position"),
        "u_position": data.get("target_u_position"),
        "active": True  # target is always active
    }
    rows.append(target)

 # -------------------------
    # Extract COMPARISON stars
    # -------------------------
    comp_pattern = re.compile(r"comparison_(\d+)_(.+)")

    comp_dict = {}

    for key, value in data.items():
        match = comp_pattern.match(key)
        if match:
            comp_id = int(match.group(1))
            field = match.group(2)

            if comp_id not in comp_dict:
                comp_dict[comp_id] = {"run": run_name,
                                      "star_type": "comparison",
                                      "star_id": comp_id}

            # Normalize field names
            if field == "x_position":
                comp_dict[comp_id]["x"] = value
            elif field == "y_position":
                comp_dict[comp_id]["y"] = value
            elif field == "aperture":
                comp_dict[comp_id]["aperture"] = value
            elif field == "active":
                comp_dict[comp_id]["active"] = value
            elif field == "r_position":
                comp_dict[comp_id]["r_position"] = value
            elif field == "u_position":
                comp_dict[comp_id]["u_position"] = value
            else:
                # Store any extra fields without losing information
                comp_dict[comp_id][field] = value

    # Add comparison stars to rows
    for comp_id, comp_data in comp_dict.items():
        # Default inactive if not specified
        comp_data.setdefault("active", False)
        rows.append(comp_data)

    return rows
    

In [5]:
def extract_all_logs(root_folder):
    """Walk through all PHOTOMETRY_n folders and extract log.yaml info."""
    all_rows = []

    for item in os.listdir(root_folder):
        if item.startswith("PHOTOMETRY_"):
            run_path = os.path.join(root_folder, item)
            log_path = os.path.join(run_path, "log.yaml")

            if os.path.exists(log_path):
                rows = extract_star_info_from_log(log_path, item)
                all_rows.extend(rows)

    return pd.DataFrame(all_rows)

In [8]:
# -------------------------
# Example usage
# -------------------------

root = "/Users/danielbhuglah/Library/CloudStorage/OneDrive-TheOpenUniversity/SXS841 OSO Observations/2026_02_24_files_consolidated/"
df = extract_all_logs(root)

output_path = downloads / "hops_photometry_log_export.xlsx"
df.to_excel(output_path, index=False)
print(f"Saved to: {output_path}")

Saved to: /Users/danielbhuglah/Library/CloudStorage/OneDrive-TheOpenUniversity/SXS841 OSO Observations/2026_02_24_files_consolidated/hops_photometry_log_export.xlsx
